# SmartCrop — Notebook 03: Feature Engineering

**Target:** `yield_t_ha`  
**Principle:** add a feature only when it has an agronomic reading and the API can rebuild it from the raw inputs.


## 1. Introduction

The clean file already stores seasonal rainfall, seasonal temperatures, soil measures, fertilizer per hectare, and a planting-day index. Copying those columns under new names would not add information. This notebook adds three contrasts that the saved model can recompute at prediction time.


## 2. Objectives

1. Document every model input, including the columns that were deliberately not duplicated.
2. Add `temperature_range_c`, `degrees_above_33_c`, and `planting_delay_days`.
3. Export `reports/tables/feature_dictionary.xlsx`.


## 3. Dataset Description

The input is `clean_dataset.csv`. Source columns keep their units from the schema. Engineered columns are differences or clipped delays of those source columns. `area_ha` is not a yield feature. Production, when needed, is `yield_t_ha × area_ha` and is not used to predict yield.


## 4. Methodology

`temperature_range_c` is maximum minus minimum temperature. `degrees_above_33_c` is how far the maximum sits above 33°C, or zero when it does not. `planting_delay_days` is how many days the planting index falls after day 40, or zero when planting is earlier. Day 1 is 1 November, so day 40 is 10 December.

No critical-period rainfall feature is created. The file has one seasonal rainfall total, not dekadal rainfall. Inventing a critical-period column would pretend the file contains a split it does not have.


## 5. Implementation


In [1]:
from pathlib import Path
import sys
import pandas as pd

def find_root() -> Path:
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    if (start / "data-science").is_dir():
        candidates.insert(0, start / "data-science")
    for candidate in candidates:
        if (candidate / "src" / "data" / "schema.py").is_file():
            return candidate
    raise FileNotFoundError("Open this notebook from the SmartCrop repository.")

ROOT = find_root()
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 120)

from data.schema import SYNTHETIC_DISCLAIMER
from features.engineering import (
    ENGINEERED_FEATURES,
    MODEL_FEATURES,
    RAW_FEATURES,
    add_engineered_features,
    feature_dictionary,
    write_feature_dictionary,
)

print(SYNTHETIC_DISCLAIMER)
clean = pd.read_csv(ROOT / "data" / "processed" / "clean_dataset.csv")
featured = add_engineered_features(clean)
dictionary_path = write_feature_dictionary(ROOT)
featured[MODEL_FEATURES + ["yield_t_ha"]].head()


SYNTHETIC DATA. These rows were generated by the SmartCrop synthetic generator for software and model-development exercises. They are not measurements from Macheke, Mashonaland East, Zimbabwe, and they must not be cited as agricultural statistics.


,rainfall_mm,avg_temperature_c,max_temperature_c,min_temperature_c,soil_ph,soil_nitrogen,soil_organic_carbon,fertilizer_kg_ha,planting_day,temperature_range_c,degrees_above_33_c,planting_delay_days,yield_t_ha
0,653.6,23.22,31.40,17.21,5.20,0.082,0.87,123.5,14.0,14.19,0.00,0.0,2.273
1,643.5,23.40,33.75,16.93,5.15,0.105,0.60,103.6,76.0,16.82,0.75,36.0,1.563
2,662.8,24.27,31.62,18.33,5.19,0.099,0.61,0.0,33.0,13.29,0.00,0.0,1.808
3,532.6,NaN,32.31,16.27,5.26,0.071,0.85,56.3,40.0,16.04,0.00,0.0,1.489
4,668.2,NaN,37.88,17.35,5.41,0.077,0.93,153.5,5.0,20.53,4.88,0.0,1.774


In [2]:
dictionary = feature_dictionary()
display(dictionary)
print("Raw inputs:", ", ".join(RAW_FEATURES))
print("Engineered:", ", ".join(ENGINEERED_FEATURES))
print(f"Feature dictionary: {dictionary_path}")
print(featured[ENGINEERED_FEATURES].describe().round(3))


,feature,kind,unit,definition
0,rainfall_mm,source,mm,Seasonal rainfall total already stored on the ...
1,avg_temperature_c,source,°C,Average temperature over the growing season. U...
2,max_temperature_c,source,°C,Mean daily maximum temperature over the growin...
3,min_temperature_c,source,°C,Mean daily minimum temperature over the growin...
4,soil_ph,source,pH,Topsoil pH.
5,soil_nitrogen,source,%,"Total soil nitrogen, percent by mass."
6,soil_organic_carbon,source,%,"Soil organic carbon, percent."
7,fertilizer_kg_ha,source,kg/ha,Fertilizer product already expressed per hecta...
8,planting_day,source,day index,Planting day index where 1 is 1 November.
9,temperature_range_c,engineered,°C,max_temperature_c minus min_temperature_c. Sum...


Raw inputs: rainfall_mm, avg_temperature_c, max_temperature_c, min_temperature_c, soil_ph, soil_nitrogen, soil_organic_carbon, fertilizer_kg_ha, planting_day
Engineered: temperature_range_c, degrees_above_33_c, planting_delay_days
Feature dictionary: /home/marvelous/.dev/Tait/capstone/data-science/reports/tables/feature_dictionary.xlsx
       temperature_range_c  degrees_above_33_c  planting_delay_days
count              883.000             913.000              910.000
mean                15.531               0.530                5.208
std                  1.993               1.183               11.159
min                 11.570               0.000                0.000
25%                 14.130               0.000                0.000
50%                 15.340               0.000                0.000
75%                 16.610               0.480                0.000
max                 23.830               6.790               60.000


## 6. Results

The dictionary has one row per model column. Engineered columns are present on the in-memory frame. They are not stored as a second copy of rainfall or fertilizer.


## 7. Visualizations

Feature distributions are part of Notebook 04, where they can be read next to yield.


## 8. Interpretation

A wider temperature range, heat above 33°C, and planting after 10 December are readable contrasts. Whether they help the model is an evaluation question, not something this notebook assumes. Soil and fertilizer stay in their original units.


## 9. Conclusion

The model matrix is the nine raw inputs plus three engineered columns. The prediction API will send the nine raw inputs. The saved pipeline rebuilds the three engineered columns, so a client cannot forget them or send a conflicting copy.
